# M6.1 — Workbook dry-run planner

Plan: [`plans/milestone_06/06_sequence_generation_plan.md`](../../plans/milestone_06/06_sequence_generation_plan.md).  
Next: `06_2_generate_sequences_no_cache.ipynb`.

This notebook exercises workbook + dry-run planning only. It does **not** run M5D, source deposition, particle-source computation, diffusion, Netgen/NGSolve, camera capture, image writing, or cache persistence.

Shows:

1. workbook loading and SHA256 provenance,
2. typed generation-plan validation (including `n_particles` / `particle_group_id`),
3. clean/particle/diffusion grouping,
4. expected cache hits and misses,
5. deterministic camera task expansion,
6. a **multi-particle group** dry-run from `configs/m6/m6_multi_particle.xlsx` (planning only).

`optical_setups.source_intensity` is a required Excel column (default `1.0`). It is recorded on each job and applied as `PointLightConfig.intensity` when sequences are generated. There is no detector-gain column: camera images are collected ray flux. M8 keeps `source_intensity` fixed across optical regimes.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
import pandas as pd
from IPython.display import display

from gummybear.datasets.generation_plan import (
    build_execution_plan,
    summarize_execution_plan,
    validate_generation_plan,
)
from gummybear.datasets.generation_workbook import load_generation_workbook
from gummybear_validation.milestone_06 import (
    cache_plan_rows,
    camera_task_rows,
    inspect_cache_plan,
    jobs_summary_table,
    workbook_sheet_summary,
)

WORKBOOK_PATH = ROOT / "configs" / "m6" / "m6_generation_plan.xlsx"
MULTI_WORKBOOK_PATH = ROOT / "configs" / "m6" / "m6_multi_particle.xlsx"


## 2. Load and inspect the workbook

Loading normalizes cells and enabled flags, verifies required sheets and columns, and records the workbook SHA256. It does not execute forward physics.


In [3]:
workbook = load_generation_workbook(WORKBOOK_PATH)
print(f"Workbook: {display_path(workbook.path)}")
print(f"SHA256:   {workbook.sha256}")
display(workbook_sheet_summary(workbook))


Workbook: configs/m6/m6_generation_plan.xlsx
SHA256:   4e6ac865c398b1c7c60e91a80f90270c993e3356568f5e71f79532ba7a378fde


,sheet,rows,enabled_rows,disabled_rows
0,sequences,1,1,0
1,optical_setups,1,1,0
2,particles,1,1,0
3,diffusion_setups,1,1,0
4,camera_schedules,1,1,0
5,corruptions,1,0,1


## 3. Validate typed sequence jobs

Validation resolves cross-sheet IDs, hashes the STL bytes without loading the mesh, expands the simple camera schedule, and constructs clean/particle cache IDs (including `source_intensity` in the clean optical key). Diffusion settings remain provenance only.


In [4]:
plan = validate_generation_plan(workbook, repo_root=ROOT)
display(jobs_summary_table(plan.jobs))
if plan.warnings:
    display(pd.DataFrame({"warnings": plan.warnings}))


,sequence_id,split,optical_setup_id,source_intensity,particle_setup_id,particle_group_id,n_particles,diffusion_setup_id,camera_schedule_id,views,resolution,clean_cache_id,particle_cache_id
0,bear_m6_smoke_001,train,opt_smoke_backlight_001,1.0,particle_smoke_sphere_001,particle_smoke_sphere_001,1,diff_smoke_robin_001,orbit_smoke_006,6,128 x 128,83b9d7a583728c38,2b71544f544171d6


## 4. Build and summarize the dry-run execution plan

The dry run groups by clean optical cache ID, particle source cache ID, diffusion-settings provenance, then ordered camera tasks. Cache probing is read-only: Phase 1 never writes cache payloads.


In [5]:
execution_plan = build_execution_plan(
    plan,
    limit=1,
    cache_root=ROOT / "data" / "generated" / "m6_2" / "_cache",
)
summary = summarize_execution_plan(
    execution_plan,
    disabled_sequence_count=len(plan.disabled_sequence_ids),
)
display(pd.DataFrame([summary.to_dict()]).T.rename(columns={0: "value"}))

cache_plan = inspect_cache_plan(execution_plan)
display(cache_plan_rows(execution_plan))
print("Plans diffusion operator cache:", cache_plan["plans_diffusion_operator_cache"])


,value
workbook_path,configs/m6/m6_generation_plan.xlsx
workbook_sha256,4e6ac865c398b1c7c60e91a80f90270c993e3356568f5e...
enabled_sequence_count,1
disabled_sequence_count,0
clean_group_count,1
particle_group_count,1
diffusion_group_count,1
sequence_count,1
frame_count,6
expected_clean_cache_hits,0


,optical_setup_id,clean_cache_id,clean_status,particle_setup_id,particle_group_id,particle_count,particle_cache_id,particle_status,diffusion_setup_id,job_count,frame_count
0,opt_smoke_backlight_001,83b9d7a583728c38,miss,particle_smoke_sphere_001,particle_smoke_sphere_001,1,2b71544f544171d6,miss,diff_smoke_robin_001,1,6


Plans diffusion operator cache: False


## 5. Inspect deterministic camera tasks

These are planning records only. No camera rays or images are generated.


In [6]:
display(camera_task_rows(execution_plan))


,sequence_id,frame_index,angle_deg,resolution_x,resolution_y
0,bear_m6_smoke_001,0,0.0,128,128
1,bear_m6_smoke_001,1,60.0,128,128
2,bear_m6_smoke_001,2,120.0,128,128
3,bear_m6_smoke_001,3,180.0,128,128
4,bear_m6_smoke_001,4,240.0,128,128
5,bear_m6_smoke_001,5,300.0,128,128


## 6. Phase 1 contract checks (single-particle smoke)

These assertions confirm the smoke workbook expands to one 128 × 128, six-view sequence and that no diffusion-operator cache is planned. The smoke job remains a **one-particle** group.


In [7]:
assert summary.enabled_sequence_count == 1
assert summary.clean_group_count == 1
assert summary.particle_group_count == 1
assert summary.diffusion_group_count == 1
assert summary.frame_count == 6
assert summary.resolutions == ((128, 128),)
assert summary.plans_diffusion_operator_cache is False
assert execution_plan.plans_operator_cache is False
assert len(plan.jobs[0].particles) == 1
assert plan.jobs[0].particle_group_id == plan.jobs[0].particle.particle_setup_id
print("Phase 1 dry-run checks passed. No forward physics was executed.")


Phase 1 dry-run checks passed. No forward physics was executed.


## 7. Multi-particle group dry-run (planning only)

Single-particle smoke remains the default (`configs/m6/m6_generation_plan.xlsx`).
For multi-particle planning, load:

```text
configs/m6/m6_multi_particle.xlsx
```

It defines an ordered `particle_group_id` of fixed, non-overlapping spheres.
Centres are authored offline in Excel; runtime placement stays `fixed`.
Package helpers (`attach_particle_group`, `write_multi_particle_generation_workbook`)
remain available to regenerate or extend this workbook — this notebook only loads it.


In [8]:
multi_workbook = load_generation_workbook(MULTI_WORKBOOK_PATH)
multi_plan = validate_generation_plan(multi_workbook, repo_root=ROOT)
multi_job = multi_plan.jobs[0]

assert len(multi_job.particles) == 2
assert multi_job.particle_group_id == "dryrun_two_sphere"
assert multi_job.sequence_id == "bear_m6_multi_001"
assert multi_job.particle_source_cache_id != plan.jobs[0].particle_source_cache_id

display(pd.DataFrame(
    [
        {
            "index": index,
            "particle_setup_id": item.particle_setup_id,
            "center_x": item.center_x,
            "center_y": item.center_y,
            "center_z": item.center_z,
            "radius": item.radius,
            "placement_mode": item.placement_mode,
        }
        for index, item in enumerate(multi_job.particles)
    ]
))
print("multi-particle dry-run checks passed (planning only)")
print("workbook:", display_path(MULTI_WORKBOOK_PATH))


,index,particle_setup_id,center_x,center_y,center_z,radius,placement_mode
0,0,dryrun_two_sphere_p000,-5.0,0.5,2.5,3.0,fixed
1,1,dryrun_two_sphere_p001,5.0,-0.5,2.5,3.0,fixed


multi-particle dry-run checks passed (planning only)
workbook: configs/m6/m6_multi_particle.xlsx
